In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
RAW_DATA_PATH = '../data/raw/heart.csv'
PROCESSED_TRAIN_PATH = '../data/processed/train.csv'
PROCESSED_TEST_PATH = '../data/processed/test.csv'

In [8]:
try:
    df = pd.read_csv(RAW_DATA_PATH)
    print("✅ Raw data loaded successfully for preprocessing.")
except FileNotFoundError:
    print(f"❌ Error: The file was not found at {RAW_DATA_PATH}")

✅ Raw data loaded successfully for preprocessing.


### Preprocessing 

####  One-Hot Encoding Categorical Features

In [11]:
# --- Preprocessing Step 1: One-Hot Encoding ---
# As identified during exploration, 'cp', 'thal', 'restecg', 'slope' are nominal categorical features.
# We convert them into a numerical format that models can understand.

print("Original shape:", df.shape)
print("Original columns:", df.columns.tolist())

categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
df_processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

print("\nShape after one-hot encoding:", df_processed.shape)
print("Columns after encoding:", df_processed.columns.tolist())

Original shape: (920, 16)
Original columns: ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']

Shape after one-hot encoding: (920, 23)
Columns after encoding: ['id', 'age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num', 'sex_Male', 'dataset_Hungary', 'dataset_Switzerland', 'dataset_VA Long Beach', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina', 'fbs_True', 'restecg_normal', 'restecg_st-t abnormality', 'exang_True', 'slope_flat', 'slope_upsloping', 'thal_normal', 'thal_reversable defect']


#### Scaling Numerical Features

In [ ]:
# --- Preprocessing Step 2: Scaling Numerical Features ---
# We standardize numerical columns to give them a mean of 0 and a standard deviation of 1.
# This helps distance-based algorithms (like SVM) and regression models perform better.

scaler = StandardScaler()
numerical_cols = df.select_dtypes(include='number').columns.tolist()
df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

print("✅ Numerical features scaled successfully.")
print("\nFirst 5 rows of the fully preprocessed data:")
display(df_processed.head())

KeyError: "['thalach'] not in index"

#### Split Data into Training and Testing Sets

In [ ]:
# --- Data Splitting ---
# We separate the data into features (X) and the target variable (y).
# Then we split them into training and testing sets to prepare for model building.
# `stratify=y` is crucial for ensuring the class distribution is the same in both sets, which is important for imbalanced targets.

X = df_processed.drop('target', axis=1)
y = df_processed['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,       # 20% of data will be used for testing
    random_state=42,     # Ensures reproducibility
    stratify=y           # Preserves the target distribution
)

print(f"Training set shape: X_train -> {X_train.shape}, y_train -> {y_train.shape}")
print(f"Testing set shape:  X_test -> {X_test.shape}, y_test -> {y_test.shape}")

### Save the Processed Data

In [ ]:
# --- Save the Processed Data ---
# For easier use in the next notebook, we'll concatenate the features (X) and target (y)
# for both the training and testing sets before saving.

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

try:
    train_df.to_csv(PROCESSED_TRAIN_PATH, index=False)
    test_df.to_csv(PROCESSED_TEST_PATH, index=False)
    print(f"\n✅ Processed training data saved to: {PROCESSED_TRAIN_PATH}")
    print(f"✅ Processed testing data saved to: {PROCESSED_TEST_PATH}")
except Exception as e:
    print(f"\n❌ An error occurred while saving the data: {e}")